[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/dynamo/dynamo.ipynb)

# A Dynamo in a Planet's Core

This example computes a dynamo: liquid metal stirred in a planet's core turning a weak seed magnetic field into a strong one, until the field pushes back on the flow that makes it. A conducting fluid drags the field's lines along with it, and a flow that stretches and twists them strengthens the field faster than the fluid's resistance lets it fade; the currents the moving fluid drives then feel the field's force, which brakes the flow, and flow and field settle into trading energy back and forth. As convection stirs Earth's liquid outer core, between its solid inner core and the mantle, a stirring here pulls a fluid shell toward a helical flow.

Spacetime, `x+y+z+t-`, holds the ball in one instant: its points are directions of space, and the magnetic field is a plane of space, the curl of a potential on the mesh's edges, so it has no sources however it grows. The moving fluid reads the magnetic plane as an electric field, the plane met with the velocity and swept through time, `(velocity & magnetic) ^ mv.t`, and drives a current with it. The field's force on the fluid is the adjugate of that motional map applied to the current: by the adjugate's defining identity, the force's power on any velocity is the current paired with that velocity's motional field, so the energy the field gains is exactly the energy the flow loses. The flow itself is a few dozen modes of the fluid shell, which carry one another along.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator
from itertools import islice, tee

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA as ga
from numga.sparse import SparseExtensor, spdiag
from examples.util import shapes
from examples.util.animation import save_figure, show_animation
from examples.util.topology import Topology
from examples.electromagnetism.dynamo import render, scenarios

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
# The planes of space, which read a point's coordinates; a function's gradient in a tetrahedron is one.
Plane = ga.gatype.from_blades("x y z")              # Vector+
# The points of space, each the direction from the origin to it; edges, velocities and currents are points too.
Point = Plane.dual().output_gatype                  # Antivector-
# Electric and magnetic fields, planes through time and planes of space.
Field = ga.gatype.bivector()                        # Bivector+
# The magnetic field alone, a plane of space.
Magnetic = Field.derive.within("x y z")             # Bivector+
# A rotation rate, or the fluid's vorticity: a plane of space.
Rate = Field.derive.within("x y z")                 # Bivector+
# A function integrated over a share of the ball is an amount.
Amount = ga.gatype.antiscalar()                     # Antiscalar-
# The observer's direction of time.
time = -mv.t.dual()                                 # [] Point-
# The electric field along a direction: the plane it sweeps through time, met with the dual.
along = (Point & time).dual()                       # [] Field+ <- Point-

## The ball in spacetime

The ball is a sphere swept outward in shells, each a subdivided icosahedron, the innermost coned to the centre: tetrahedra whose edges are directions of space. A corner's hat function rises from nothing on the face across from it to one at the corner, and its gradient is the volume that face sweeps through time over the prism the tetrahedron sweeps, `(face_patches & time) / (3 * prisms)`. Whitney's form of an edge blends the gradients at its ends, so a potential on the edges is a plane in every tetrahedron, and its curl, twice the wedge of the end gradients, is the magnetic field. Twice the magnetic energy is the field paired with its dual over each volume, pulled back to the edges by the curl's adjugate.

In [ ]:
# The sphere's subdivision levels, the ball's shells, and the mantle's outer radius, the core's radius being one.
n_levels = 2
n_shells = 6
mantle_radius = 1.5

In [ ]:
class Ball:
    """A tetrahedral ball's Whitney operators: the potential's curl, the magnetic energy, and the electric field each
    edge's potential sets up in each tetrahedron."""

    def __init__(self, topology: Topology, vertices: Point):
        self.topology = topology
        self.centres = topology.incidence[3, 0].mean(axis=0) * vertices          # Point-[tets]
        self.edges = topology.d[0] * vertices                                    # Point-[edges]
        # Each tetrahedron's edges in its own vertex order; the first three run from its first corner.
        edges = topology.oriented[3, 1] * self.edges                             # [tet_edges] Point-[tets]
        volumes = (edges[0] & edges[1] & edges[2]) / 6                           # Antitrivector-[tets]
        # The prism each tetrahedron sweeps in a unit of time; its size is the tetrahedron's volume.
        self.prisms = volumes & time                                             # Scalar-[tets]
        sides = topology.boundary[2, 1] * self.edges                             # [3] Point-[faces]
        face_patches = topology.boundary[3, 2] * ((sides[0] & sides[1]) / 2)     # [corners] Patch-[tets]
        # A corner hat's gradient is the volume its opposite face sweeps through time over the prism.
        corner_gradients = (face_patches & time) / (3 * self.prisms)             # [corners] Plane+[tets]
        tails, heads = topology.corners[3, 1]
        # At each centroid an edge's Whitney form is a quarter of the difference of its ends' gradients, and its curl
        # is twice their wedge.
        centroid_forms = (corner_gradients[heads] - corner_gradients[tails]) / 4    # [tet_edges] Plane+[tets]
        self.whitney = (spdiag(centroid_forms * Scalar) * topology.oriented[3, 1]).sum(axis=0)   # (Plane+ <- Scalar+)[tets <- edges]
        curls = 2 * (corner_gradients[tails] ^ corner_gradients[heads])          # [tet_edges] Magnetic+[tets]
        self.curl = (spdiag(curls * Scalar) * topology.oriented[3, 1]).sum(axis=0)   # (Magnetic+ <- Scalar+)[tets <- edges]
        # Each edge's potential changing in time sets up an electric field, its Whitney form swept through time.
        ones = mv.scalar(np.ones((topology.n_elements[3], 1))).field()          # Scalar+[tets]
        sweep = spdiag(ones * (Plane ^ mv.t))                                    # (Field+ <- Plane+)[tets <- tets]
        self.electric = sweep(self.whitney)                                      # (Field+ <- Scalar+)[tets <- edges]
        # Twice the magnetic energy: the field paired with its dual over each volume, pulled back to the edges.
        vacuum = spdiag(self.prisms.norm() * Magnetic.dual())                    # (Patch- <- Magnetic+)[tets <- tets]
        self.stiffness = self.curl.adjugate()(vacuum(self.curl))                 # (Amount- <- Scalar+)[edges <- edges]


topology, coordinates = shapes.ball(n_levels, n_shells)
vertices = mv(Plane, mantle_radius * coordinates).dual().field()                 # Point-[vertices]
ball = Ball(topology, vertices)

## The flow's modes

The fluid is the shell between the inner core and the mantle. A flow there is its circulation along the shell's edges: their Whitney forms are its momentum in each tetrahedron, the momentum's dual its velocity, and their curl its vorticity. An incompressible flow pours nothing into any vertex's share of the fluid, and taking away the gradient of the pressure that would carry that inflow makes any flow incompressible, one solve with the pressure's stiffness. The shell's slowest scalar modes, laid along each axis and made incompressible, span smooth flows; among them the flow's modes are those of least enstrophy for their kinetic energy, the eigenmodes of the enstrophy form against the energy form, orthonormal in kinetic energy, each eigenvalue a mode's rate of viscous decay.

In [ ]:
# The fluid's slowest scalar modes the flow's modes are built from, the flow's modes, and those drawn, from the slowest
# to a fine one.
n_scalar_modes = 80
n_modes = 60
shown_modes = np.array([0, 3, 8, 40])

In [ ]:
class Flow:
    """A flow in the fluid part of the ball as the amplitudes of its slowest modes, orthonormal in kinetic energy, from
    the fluid's slowest scalar modes laid along each axis and made incompressible."""

    def __init__(self, fluid: Ball, embedding: SparseExtensor, n_scalar_modes: int, n_modes: int):
        topology, volumes = fluid.topology, fluid.prisms.norm()                 # Scalar+[fluid]
        ones = mv.scalar(np.ones((topology.n_elements[3], 1))).field()          # Scalar+[fluid]
        circulating = spdiag(ones * Plane.dual())(fluid.whitney)                 # (Point- <- Scalar+)[fluid <- edges]
        momentum = spdiag(volumes * Point.dual())                                # (Plane+ <- Point-)[fluid <- fluid]
        kinetic = circulating.adjugate()(momentum(circulating))                  # (Amount- <- Scalar+)[edges <- edges]
        # The fluid flowing into each vertex's share of the fluid, and the stiffness of a pressure pushing it down its
        # gradient, with the pressure's free constant pinned at the first vertex.
        inflow = ~topology.d[0] * kinetic                                        # (Amount- <- Scalar+)[vertices <- edges]
        pressure = inflow * topology.d[0]                                        # (Amount- <- Scalar+)[vertices <- vertices]
        reference = mv.scalar((np.arange(topology.n_vertices) == 0)[:, None]).field()   # Scalar+[vertices]
        pinned = pressure + spdiag(pressure.diagonal() * reference)              # (Amount- <- Scalar+)[vertices <- vertices]
        shares = spdiag((~topology.incidence[3, 0].mean(axis=0) * volumes) * Scalar.dual())   # (Amount- <- Scalar+)[vertices <- vertices]
        _, scalar_modes = pressure.eigh(shares, n_scalar_modes + 1)              # [scalar + 1] Scalar+[vertices]
        # Each scalar mode but the constant, at the edges' midpoints, times each edge's reach along each axis.
        means = topology.incidence[1, 0].mean(axis=0) * scalar_modes[1:]         # [scalar] Scalar+[edges]
        reaches = stack((mv.x, mv.y, mv.z))[:, None] & fluid.edges                # [3, 1] Scalar+[edges]
        seeds = (means * reaches).reshape(-1)                                    # [seeds] Scalar+[edges]
        incompressible = seeds - topology.d[0] * pinned.solve(inflow(seeds))     # [seeds] Scalar+[edges]
        # Twice the kinetic energy and twice the enstrophy of the flows the seeds span, as forms on their mixtures.
        spanned = (incompressible * Scalar).batch().field(0, 1).adjoint()        # (Scalar+ <- Scalar+)[edges <- seeds]
        energy = (Scalar & (spanned.adjugate()(kinetic(incompressible)).batch() * Scalar)).field(1, 2)   # Scalar+ <- (Scalar+[seeds], Scalar+[seeds])
        enstrophy = (Scalar & (spanned.adjugate()(fluid.stiffness(incompressible)).batch() * Scalar)).field(1, 2)   # Scalar+ <- (Scalar+[seeds], Scalar+[seeds])
        decay, mixtures = enstrophy.eigh(energy)                                 # [seeds] Scalar+, [seeds] Scalar+[seeds]
        self.decay = decay[:n_modes].field()                                     # Scalar+[modes]
        self.modes = spanned(mixtures[:n_modes])                                 # [modes] Scalar+[edges]
        self.circulation = (self.modes * Scalar).batch().field(0, 1).adjoint()  # (Scalar+ <- Scalar+)[edges <- modes]
        # The velocity and the vorticity of the fluid's circulations, placed among the ball's tetrahedra.
        self.velocity = embedding * circulating                                  # (Point- <- Scalar+)[tets <- edges]
        self.vorticity = embedding * fluid.curl                                  # (Rate+ <- Scalar+)[tets <- edges]

    def reading(self, form: Plane) -> Scalar:
        """Each mode's share of a form on the velocities in the ball's tetrahedra, `Scalar+[modes]`: the power a force
        puts into each mode, or a momentum's amplitude in each."""
        return (self.modes & self.velocity.adjugate()(form)).sites.sum().field()


# The fluid shell's tetrahedra on its own vertices, and the map placing a field on them among the ball's.
fluid_topology, fluid_vertices, embedding = scenarios.outer_core(topology, vertices)
flow = Flow(Ball(fluid_topology, fluid_vertices), embedding, n_scalar_modes, n_modes)
save_figure(render.draw_modes(ball, mantle_radius, flow.velocity(flow.modes[shown_modes])), "dynamo_modes");

Each panel cuts the ball open, the cut face coloured by the mode's speed, and stands the mode's streamlines in front of it. The slowest mode turns the shell nearly rigidly about the still inner core; the finer ones wind into knots.

## Stirring and the field's force

A potential changes by `conductance(dA/dt) == drive(A) - stiffness(A)`: the conductance meets the electric field a changing potential sets up, the drive is the moving fluid's motional field pushing current along the edges, and the stiffness is the field's tension. The flow's momentum changes against its velocity met with its vorticity plane, `-(velocity & vorticity)`, which paired with the velocity itself vanishes: carrying itself, the flow moves energy between its modes and makes none. The stirring pulls the flow back toward a helical flow, two spheromaks turning about axes square to each other, at a fixed rate. The field is stepped with the stirred flow's drive inside one factorized map and the flow's departure from it explicitly, and the current of each step, the one Ampère's law balances, brakes the flow halfway through it.

In [ ]:
def ohm(ball: Ball, conductivity: Scalar) -> SparseExtensor:
    """Ohm's law in each tetrahedron: the current a field drives, the field read back along the direction it points,
    swept through time and weighted by the volume, so that a field paired with it is the heat it makes."""
    return spdiag(ball.prisms.norm() * conductivity * (along.inverse() & time))  # (Patch- <- Field+)[tets <- tets]


def induction(ball: Ball, velocities: Point, conductivity: Scalar) -> tuple[SparseExtensor, SparseExtensor]:
    """The conductance a changing potential meets and the drive of the moving fluid's electric field, both on the
    edges."""
    current = ohm(ball, conductivity)                                            # (Patch- <- Field+)[tets <- tets]
    conductance = ball.electric.adjugate()(current(ball.electric))               # (Amount- <- Scalar+)[edges <- edges]
    # The moving fluid reads the magnetic plane as an electric field: met with the velocity, swept through time.
    motional = spdiag((velocities & Magnetic) ^ mv.t)(ball.curl)                 # (Field+ <- Scalar+)[tets <- edges]
    drive = ball.electric.adjugate()(current(motional))                          # (Amount- <- Scalar+)[edges <- edges]
    return conductance, drive

In [ ]:
def advected(ball: Ball, flow: Flow, stirred: Point, amplitudes: Scalar, potential: Scalar) -> tuple[Field, Scalar]:
    """The flow's own part of a step: the motional field of its departure from the stirred flow, `Field+[tets]`, and
    the rate of each mode's amplitude as the flow carries itself, `Scalar+[modes]`."""
    circulations = flow.circulation(amplitudes)                                 # Scalar+[edges]
    velocity, vorticity = flow.velocity(circulations), flow.vorticity(circulations)   # Point-[tets], Rate+[tets]
    departure = spdiag((Point & ball.curl(potential)) ^ mv.t)(velocity - stirred)   # Field+[tets]
    return departure, flow.reading(-(ball.prisms.norm() * (velocity & vorticity)))


def dynamo(ball: Ball, flow: Flow, conductivity: Scalar, viscosity: float, pull: float, amplitudes: Scalar,
           potential: Scalar, dt: float, n_steps: int) -> Iterator[tuple[Scalar, Scalar]]:
    """Successive flow amplitudes and potentials, `Scalar+[modes]` and `Scalar+[edges]`, from the stirred flow
    `amplitudes`, which the stirring pulls the flow back toward at the rate `pull`: Crank–Nicolson steps under the
    field's diffusion and the stirred flow's drive, the flow's viscosity and the stirring's pull, and Adams–Bashforth
    steps under the rest."""
    current = ohm(ball, conductivity)                                            # (Patch- <- Field+)[tets <- tets]
    stirred = flow.velocity(flow.circulation(amplitudes))                       # Point-[tets]
    conductance, drive = induction(ball, stirred, conductivity)
    growth = drive - ball.stiffness                                              # (Amount- <- Scalar+)[edges <- edges]
    ahead, behind = conductance - (dt / 2) * growth, conductance + (dt / 2) * growth
    damping = viscosity * flow.decay + pull                                      # Scalar+[modes]
    stirring = pull * amplitudes                                                 # Scalar+[modes]
    departure, rate = advected(ball, flow, stirred, amplitudes, potential)       # Field+[tets], Scalar+[modes]
    previous_departure, previous_rate = departure, rate                          # Field+[tets], Scalar+[modes]
    yield amplitudes, potential
    for _ in range(n_steps):
        pushed = 1.5 * departure - 0.5 * previous_departure                      # Field+[tets]
        stepped = ahead.solve(behind(potential) + dt * ball.electric.adjugate()(current(pushed)))   # Scalar+[edges]
        change, middle = (stepped - potential) / dt, (stepped + potential) / 2   # Scalar+[edges], Scalar+[edges]
        # The current of the step, the one Ampère's law balances, `electric.adjugate()(currents) == stiffness(middle)`:
        # the motional field less the field the changing potential sets up.
        motional = spdiag((Point & ball.curl(middle)) ^ mv.t)                   # (Field+ <- Point-)[tets <- tets]
        currents = current(motional(stirred) + pushed - ball.electric(change))  # Patch-[tets]
        # The field's force on the current is the adjugate of the motional map, which reads the current as a form on
        # velocities.
        braking = flow.reading(-motional.adjugate()(currents))                   # Scalar+[modes]
        pushed_flow = amplitudes + dt * (1.5 * rate - 0.5 * previous_rate + braking + stirring - damping * amplitudes / 2)   # Scalar+[modes]
        amplitudes, potential = pushed_flow / (1 + (dt / 2) * damping), stepped  # Scalar+[modes], Scalar+[edges]
        previous_departure, previous_rate = departure, rate
        departure, rate = advected(ball, flow, stirred, amplitudes, potential)
        yield amplitudes, potential

## The dynamo

The core conducts well and the mantle a hundredth as well, and the seed is a weak uniform field. The stirred flow stretches and twists the field until it is strong enough to brake the flow; the animation starts there, where the field saturates, and shows the field lines in white and the flow's streamlines in cyan, with chevrons pointing downstream, seen from above inside the core's surface, a faint translucent sphere. This cell takes about a minute, most of it the steps and the drawing; its frames show as they are computed.

In [ ]:
# The magnetic Reynolds number, the core's conductivity for a flow whose fastest point moves at unit speed, and how much
# poorer a conductor the mantle is; the fluid's viscosity and the rate of the stirring's pull; the seed field.
reynolds = 160.0
mantle_conductance = 1e-2
viscosity = 0.01
pull = 1.0
seed_field = mv.xy * 1e-2 + mv.yz * 3e-3                                       # [] Magnetic+
# The step, the steps run, those before the animation starts, the stride between frames, and the frame duration.
dt = 0.05
n_steps = 2400
n_skipped_steps = 1200
stride = 10
frame_duration_ms = 40

In [ ]:
radii = ball.centres.squared().abs().square_root()                               # Scalar+[tets]
conductivity = mv.scalar(np.where((radii < 1).reshape(-1), reynolds, reynolds * mantle_conductance)[:, None]).field()   # Scalar+[tets]
# The stirred flow, the helical flow's share in each mode, and the seed field's potential.
stirred = flow.reading(ball.prisms.norm() * scenarios.helical_flow(topology, vertices).dual())   # Scalar+[modes]
potential = scenarios.seed(topology, vertices, seed_field)                       # Scalar+[edges]
# The run, drawn from above and cut away as it comes, and kept whole for the energies.
above, cut_open, kept = tee(dynamo(ball, flow, conductivity, viscosity, pull, stirred, potential, dt, n_steps), 3)


def frames(states: Iterable[tuple[Scalar, Scalar]]) -> Iterator[tuple[Magnetic, Point]]:
    """The field and the flow in each tetrahedron, `Magnetic+[tets]` and `Point-[tets]`, of every stride-th state of the
    saturated dynamo."""
    for amplitudes, potential in islice(states, n_skipped_steps, None, stride):
        yield ball.curl(potential), flow.velocity(flow.circulation(amplitudes))


n_frames = (n_steps - n_skipped_steps) // stride + 1
show_animation(render.animate_lines(ball, frames(above)), "dynamo_lines", frame_duration_ms, n_frames=n_frames)

The same run cut open by a vertical plane through the centre: the cut face is streaked along the flow within it, each streak fading downstream, and the field lines of the removed half stand in front of it. This cell takes about half a minute, drawing.

In [ ]:
show_animation(render.animate(ball, mantle_radius, frames(cut_open)), "dynamo", frame_duration_ms, n_frames=n_frames)

## Energy traded between flow and field

The modes being orthonormal in kinetic energy, the flow's is half the sum of its amplitudes' squares; the field's is half the stiffness's amount of the potential, paired with the potential. The field grows by orders of magnitude while the flow hardly notices, until it is strong enough to brake the flow; from there on the two trade energy back and forth.

In [ ]:
amplitudes, potentials = map(stack, zip(*islice(kept, 0, None, stride)))       # [frames] Scalar+[modes], [frames] Scalar+[edges]
kinetic = (amplitudes * amplitudes).sites.sum() / 2                              # [frames] Scalar+
magnetic = (potentials * ball.stiffness(potentials)).sites.sum() / 2             # [frames] Amount-
save_figure(render.draw_energies(np.arange(0, n_steps + 1, stride) * dt, kinetic, magnetic), "dynamo_energies");

In [ ]:
# checks
unit = mv.scalar([1.0])
# The flow's kinetic energy is half the sum of its amplitudes' squares: its momentum paired with its velocity.
velocity = flow.velocity(flow.circulation(amplitudes[-1]))                      # Point-[tets]
energy = (ball.prisms.norm() * (velocity.dual() & velocity)).sites.sum() / 2
np.testing.assert_allclose(energy.to_array(), kinetic[-1].to_array(), rtol=1e-10)
# Carrying itself, the flow moves energy between its modes and makes none.
_, rate = advected(ball, flow, flow.velocity(flow.circulation(stirred)), amplitudes[-1], potentials[-1])
scale = ((rate * rate).sites.sum() * (amplitudes[-1] * amplitudes[-1]).sites.sum()).square_root()
np.testing.assert_allclose((amplitudes[-1] * rate).sites.sum().to_array(), 0.0, atol=1e-12 * scale.to_array())
# The field has no sources: its flux out of every tetrahedron vanishes.
fluxes = topology.d[1] * potentials[-1]                                          # Scalar+[faces]
np.testing.assert_allclose((topology.d[2] * fluxes).to_array(), 0.0, atol=1e-12 * np.abs(fluxes.to_array()).max())
# The field grows by orders of magnitude, and brakes the flow.
assert ((magnetic[-1] - 100 * magnetic[0]) & unit).to_array() > 0
assert kinetic.min().to_array() < 0.75 * kinetic[0].to_array()